In [5]:
! uv pip install langchain openai tiktoken langchain-community rapidocr-onnxruntime python-dotenv google-generativeai sentence-transformers

Using Python 3.11.15 environment at: C:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv
Resolved 116 packages in 1.36s
 Downloaded shapely
 Downloaded sqlalchemy
 Downloaded pillow
 Downloaded scikit-learn
 Downloaded numpy
 Downloaded onnxruntime
 Downloaded scipy
 Downloaded torch
Prepared 20 packages in 1m 39s
Installed 63 packages in 5.31s
 + aiohappyeyeballs==2.6.1
 + aiohttp==3.13.5
 + aiosignal==1.4.0
 + annotated-doc==0.0.4
 + attrs==26.1.0
 + click==8.3.3
 + dataclasses-json==0.6.7
 + filelock==3.29.0
 + flatbuffers==25.12.19
 + frozenlist==1.8.0
 + fsspec==2026.4.0
 + greenlet==3.5.0
 + hf-xet==1.4.3
 + httpx-sse==0.4.3
 + huggingface-hub==1.13.0
 + jinja2==3.1.6
 + jiter==0.14.0
 + joblib==1.5.3
 + langchain==1.2.17
 + langchain-classic==1.0.5
 + langchain-community==0.4.1
 + langchain-text-splitters==1.1.2
 + langgraph==1.1.10
 + langgraph-checkpoint==4.0.3
 + langgraph-prebuilt==1.0.13
 + langgraph-sdk==0.3.13
 + markdown-it-py==4.0.0
 + markupsafe==3.0.3
 

In [6]:
import os
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv())

print("KEY EXISTS:", os.getenv("GOOGLE_API_KEY") is not None)

KEY EXISTS: True


## Data Ingestion

In [7]:
! uv pip install tqdm

Using Python 3.11.15 environment at: C:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv
Checked 1 package in 21ms


In [8]:
from tqdm import tqdm
import time

In [9]:
! uv pip install -U "protobuf>=4.25.3" "google-generativeai"

Using Python 3.11.15 environment at: C:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv
Resolved 31 packages in 715ms
Checked 31 packages in 2ms


In [10]:
from langchain_community.document_loaders import TextLoader
import google.generativeai as genai

In [55]:
loader=TextLoader("C:\\Users\\sbson\\OneDrive\\desktop\\llmops_project_agentic-based\\data\\Agentic_AI.txt",encoding="UTF-8")
documents=loader.load()

In [14]:
documents=loader.load()

In [15]:
documents[0].page_content[:500]

'Understanding Agentic AI\n\nAgentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.\n\nKey Characteristics of Agentic AI\n\nAgentic AI systems are distinct from traditional AI models due t'

In [56]:
#divide the documents into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=80)
text_chunks=text_splitter.split_documents(documents)

In [17]:
text_chunks[1].page_content[:50]

'Agentic AI refers to a new paradigm in artificial '

In [18]:
! uv pip install FAISS-cpu

Using Python 3.11.15 environment at: C:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv
Resolved 3 packages in 594ms
 Downloaded faiss-cpu
Prepared 1 package in 8.45s
Installed 1 package in 22ms
 + faiss-cpu==1.13.2


In [20]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

C:\Users\sbson\AppData\Local\Temp\ipykernel_6440\3496078185.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2070.22it/s]


In [57]:
# creating vector store from the text chunks
vectorstore=FAISS.from_documents(text_chunks,embeddings)

In [58]:
retriever=vectorstore.as_retriever(search_kwargs={"k": 6})

In [59]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [22]:
#data retrieval
query="what is the key characteristics of agentic AI?"
docs=vectorstore.similarity_search(query)
#display the results



In [23]:
docs = vectorstore.similarity_search_with_score(query, k=3)
docs = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(docs):
    print(f"\n🔹 Document {i+1}")
    print("Score:", score)
    print(doc.page_content)
    print("-" * 50)



🔹 Document 1
Score: 0.12955803
Key Characteristics of Agentic AI
--------------------------------------------------

🔹 Document 2
Score: 0.3544402
Understanding Agentic AI
--------------------------------------------------

🔹 Document 3
Score: 0.36738753
Applications of Agentic AI
--------------------------------------------------


In [24]:
from langchain_core.prompts import PromptTemplate
template = """
You are an intelligent AI assistant.

Use ONLY the provided context to answer the question.
If the answer is not present in the context, say:
"I don't know based on the given information."

Context:
{context}

Question:
{question}

Answer:
"""
prompt = PromptTemplate.from_template(template)

In [25]:
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nYou are an intelligent AI assistant.\n\nUse ONLY the provided context to answer the question.\nIf the answer is not present in the context, say:\n"I don\'t know based on the given information."\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:\n')

In [26]:
from langchain_core.output_parsers import StrOutputParser
output=StrOutputParser()

In [27]:
! uv pip install langchain-google-genai

Using Python 3.11.15 environment at: C:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv
Checked 1 package in 50ms


In [ ]:
#load llm model 
import os
from langchain_google_genai import ChatGoogleGenerativeAI
model_name="gemini-1.5-flash-001"

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

response = llm.invoke("Explain agentic AI")
print(response.content)


c:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sbson\AppData\Local\Temp\ipykernel_6440\3499207535.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


Using model: gemini-2.5-flash
**Agentic AI** refers to AI systems designed to act autonomously in an environment to achieve a specific goal or set of goals. Unlike traditional AI models that might simply process input and generate an output (like an LLM answering a question), an agentic AI takes initiative, plans a series of actions, executes those actions, observes the results, and adapts its plan as needed, all without continuous human prompting for each step.

Think of it as an AI that doesn't just *think* but also *does*.

### Key Characteristics of Agentic AI:

1.  **Goal-Oriented:** It's given a high-level objective (e.g., "build a website that does X," "research the market for Y," "debug this code," "book me a flight to Z").
2.  **Planning:** It breaks down the high-level goal into a sequence of smaller, manageable tasks. This often involves reasoning about the necessary steps.
3.  **Execution/Action:** It doesn't just propose actions; it *takes* them. This involves interacting 

In [28]:
print(response.content)

**Agentic AI** refers to AI systems designed to act autonomously in an environment to achieve a specific goal or set of goals. Unlike traditional AI models that might simply process input and generate an output (like an LLM answering a question), an agentic AI takes initiative, plans a series of actions, executes those actions, observes the results, and adapts its plan as needed, all without continuous human prompting for each step.

Think of it as an AI that doesn't just *think* but also *does*.

### Key Characteristics of Agentic AI:

1.  **Goal-Oriented:** It's given a high-level objective (e.g., "build a website that does X," "research the market for Y," "debug this code," "book me a flight to Z").
2.  **Planning:** It breaks down the high-level goal into a sequence of smaller, manageable tasks. This often involves reasoning about the necessary steps.
3.  **Execution/Action:** It doesn't just propose actions; it *takes* them. This involves interacting with external tools, APIs, web

In [31]:
response = llm.invoke("Is AI really dangerous?")

In [32]:
print(response.content)

The question "Is AI really dangerous?" is complex, and the answer isn't a simple yes or no. It's more accurate to say that AI, like many powerful technologies, carries significant risks and potential dangers, but also immense benefits. The danger largely depends on how it's developed, deployed, and governed.

Here's a breakdown of the potential dangers, categorized by their proximity and nature:

### Near-Term & Present Dangers (Happening now or very soon)

1.  **Bias and Discrimination:** AI systems are trained on data. If that data reflects existing societal biases (e.g., racial, gender, socioeconomic), the AI will learn and perpetuate those biases, leading to unfair outcomes in areas like hiring, lending, criminal justice, and healthcare.
2.  **Misinformation and Disinformation:** AI can generate highly convincing fake text, images, audio, and video (deepfakes). This can be used to spread propaganda, manipulate public opinion, impersonate individuals, or create chaos, undermining tr

In [36]:
from langchain_core.runnables import RunnablePassthrough

In [60]:
rag_chain=({"context": retriever | format_docs, "question": RunnablePassthrough()}
           | prompt
           | llm
           | output)

In [63]:
question = "what is Agentic AI?"
context_docs = retriever.invoke(question)


In [62]:
rag_chain.invoke("what is Agentic AI?")

'Agentic AI refers to a new paradigm in artificial intelligence where systems are designed not just to respond to queries or perform specific tasks, but to operate autonomously towards achieving predefined goals. This involves capabilities such as planning, reasoning, executing actions, and continuously adapting to dynamic environments without constant human intervention.'

In [1]:
import sys
print(sys.version)
print(sys.executable)

3.11.15 (main, Apr 14 2026, 14:31:26) [MSC v.1944 64 bit (AMD64)]
c:\Users\sbson\OneDrive\desktop\llmops_project_agentic-based\.venv\Scripts\python.exe
